In [69]:

from functions import *
import pandas as pd
#import importlib, functions
#importlib.reload(functions)



In [70]:
binary_k=0.1
continuous_k=0.3

In [71]:
# Stable binary variable (e.g., symptom yes/no)
binary_data = binary_column(
    n_patients=2000,
    start_date="2020-03-16",
    end_date="2022-12-02",
    cough_p=0.35,
    admissions_pattern="spiky",
    seed=456,
    col_name="fever",
)

events = None
results, fig = rolling_metric_fixed_baseline_cusum(
    data=binary_data, 
    value_col='fever',
    date_col="date_admit",
    events=events,
    batch=100,
    baseline_batches=3,
    n_bins=10,
    k=binary_k,              # example for a distance metric; tune from stable period
    metric_name=None,    # uses first metric returned by your metrics function
    plot=True,
    th=0.5
)

fig.show()


In [72]:
# Stable continuous variable 
continuous_data = continuous_column(
                    n_patients=2000,
    start_date="2020-03-16",
    end_date="2022-12-02",
                    
                    low=25,
                    high=75,
                    mean=50,
                    sd=15,
                    col_integer=True,
                    admissions_pattern='uniform',
                    seed=182,
                    col_name='crt',
                )

events = None
results, fig = rolling_metric_fixed_baseline_cusum(
    data=continuous_data, 
    value_col='crt',
    date_col="date_admit",
    events=events,
    batch=100,
    baseline_batches=3,
    n_bins=10,
    k=continuous_k,              # example for a distance metric; tune from stable period
    metric_name=None,    # uses first metric returned by your metrics function
    plot=True,
    th=0.5
)

fig.show()


In [75]:
events = {"2020-09-13":"Change", "2021-03-13":"Change"}
df_cont = generate_scenario(
    kind="continuous", scenario_type="gradual_drift",
    start_date="2020-03-16", end_date="2021-05-02",
    events=events, n_per_block_list=[15000,10000,10000],
    values=[95, 50, 10],  
    col_name="crp"
)


results, fig = rolling_metric_fixed_baseline_cusum(
    data=df_cont, 
    value_col='crp',
    date_col="date_admit",
    events=events,
    batch=100,
    baseline_batches=3,
    n_bins=10,
    k=0.26 ,          # example for a distance metric; tune from stable period
    metric_name=None,    # uses first metric returned by your metrics function
    plot=True,
    th=1,
    subsample_rate_low=0.2
)

fig.show()

#for i in results:
#    print(i['cur_size'])

k = np.quantile([r["metric_value"] for r in results[:10]], 0.75)
print(k)    

0.14854247856104422


In [77]:


path='C:/Users/egarcia/OneDrive - Nexus365/Projects/MsC_EBHC/Disertation/data/'
file='ISARIC_Processed_COVID19_dataset_JUL-2025.csv'

data=pd.read_csv(path+file)
data=data.loc[data['country_iso']=='GBR'] 

# convert to datetime (coerce invalid -> NaT) and sort oldest first
data['date_admit'] = pd.to_datetime(data['date_admit'], errors='coerce')
# if you want to drop rows with invalid/missing dates:
data = data.dropna(subset=['date_admit'])
data = data.sort_values('date_admit', ascending=True).reset_index(drop=True)

data = data.loc[data['date_admit'] >= '2020-01-29']
data = data.loc[data['date_admit'] <= '2022-02-28']

events = {
    "2020-03-16": "closures",
    # "2020-03-23": "lockdown",
    "2020-05-10": "Stay alert msg",
    "2020-06-08": "Quarantine for arrivals",
    "2020-12-08": "First Pfizer vaccine dose (UK)",
    #"2020-12-30": "Oxford-AstraZeneca approved",
    #"2021-01-04": "First AstraZeneca dose (UK)",
    "2021-02-15": "15M people vaccinated",
    "2021-04-01": "First Delta",
    # "2021-06-01": "Delta becomes dominant in UK",
    #"2021-04-15": "All over-50s offered first dose",
    "2021-07-19": "restrictions lifted",
    "2021-09-20": "Booster programme begins",
    "2021-11-27": "First Omicron case",
    #"2021-12-12": "Booster offered to all adults",
}



0.11716271315568622


In [78]:
results, fig = rolling_metric_fixed_baseline_cusum(
    data=data, 
    value_col='age',
    date_col="date_admit",
    events=events,
    batch=100,
    baseline_batches=3,
    n_bins=10,
    k=0.35 ,          # example for a distance metric; tune from stable period
    metric_name=None,    # uses first metric returned by your metrics function
    plot=True,
    th=1,
    subsample_rate_low=0.2
)

fig.show()
k = np.quantile([r["metric_value"] for r in results[:10]], 0.75)
print(k)   


0.14501182650807543


In [80]:
for i in data.columns:print(i)

Unnamed: 0
usubjid
studyid
siteid_final
date_admit
age
slider_sex
ethnic
slider_country
income
region
cov_det_cronavir
cov_det_sarscov2
cov_id_cronavir
cov_id_sarscov2
cov_det_id
comorbid_aids_hiv
comorbid_asthma
comorbid_chronic_cardiac_disease
comorbid_chronic_haematological_disease
comorbid_chronic_kidney_disease
comorbid_chronic_neurological_disorder
comorbid_chronic_pulmonary_disease
comorbid_dementia
comorbid_diabetes
comorbid_hypertension
comorbid_immunosuppression
comorbid_liver_disease
comorbid_malignant_neoplasm
comorbid_malnutrition
comorbid_obesity
comorbid_other
comorbid_rare_diseases_and_inborn_errors_of_metabolism
comorbid_rheumatologic_disorder
comorbid_smoking
comorbid_transplantation
comorbid_tuberculosis
comorbid_pregnancy
date_onset
clin_diag_covid_19
slider_symptomatic
symptoms_abdominal_pain
symptoms_altered_consciousness_confusion
symptoms_asymptomatic
symptoms_bleeding
symptoms_chest_pain
symptoms_conjunctivitis
symptoms_cough
symptoms_diarrhoea
symptoms_ear_pai

In [88]:
results, fig = rolling_metric_fixed_baseline_cusum(
    data=data, 
    value_col='symptoms_runny_nose',
    date_col="date_admit",
    events=events,
    batch=100,
    baseline_batches=3,
    n_bins=10,
    k=0.12 ,          # example for a distance metric; tune from stable period
    metric_name=None,    # uses first metric returned by your metrics function
    plot=True,
    th=1.5,
    subsample_rate_low=0.2
)

fig.show()
k = np.quantile([r["metric_value"] for r in results[:10]], 0.75)
print(k)   


0.124486344096824
